# Lab 6: Interaction Terms

Two sleep interventions may work differently when used together. The association between daily steps and resting heart rate may also differ between diet groups. You'll use these two simulated studies to interpret interactions, compare regression tests, and estimate uncertainty in a difference between slopes.


In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5)
source("lab6_packages.R")
if (lab6_ensure_packages()) {
    library(ottr)
    library(ggplot2)
    library(dplyr)
}


## Part 1: Interactions in a Four-Group Study

A researcher randomly assigns 40 participants to four groups, with 10 participants in each, to compare two sleep-hygiene interventions:

* **Blue light filter** (`blue_light`): use a blue-light-blocking screen filter after 9pm (`"no"` or `"yes"`).
* **Caffeine cutoff** (`caffeine_cutoff`): avoid caffeine after 3pm (`"no"` or `"yes"`).

After a month, participants rate their average nightly sleep quality from 1 to 10. These simulated scores are rounded and limited to that range. Part 2 uses a separate model with Normal errors for the testing simulations.


Run the cell below to generate the study data.


In [ ]:
set.seed(123)
n_per_group <- 10
sleep_df <- tibble(
  sleep_score = round(pmin(10, pmax(1, c(
    rnorm(n_per_group, mean = 5.0, sd = 1.2),
    rnorm(n_per_group, mean = 6.1, sd = 1.2),
    rnorm(n_per_group, mean = 6.3, sd = 1.2),
    rnorm(n_per_group, mean = 9.2, sd = 1.2)
  ))), 1),
  blue_light = factor(
    c(rep("no", n_per_group), rep("yes", n_per_group),
      rep("no", n_per_group), rep("yes", n_per_group)),
    levels = c("no", "yes")
  ),
  caffeine_cutoff = factor(
    c(rep("no", 2 * n_per_group), rep("yes", 2 * n_per_group)),
    levels = c("no", "yes")
  )
)
head(sleep_df)


**Question 1.1.** Write `summarize_sleep(data)` to return the unrounded mean sleep score for each treatment combination. Return a data frame with columns `blue_light`, `caffeine_cutoff`, and `mean_sleep_score`, in that order. Order the rows as (no, no), (no, yes), (yes, no), and (yes, yes).

Your function should work with other datasets containing these columns and all four combinations. The next cell applies it to `sleep_df`.


_Points:_ 0.5

In [ ]:
summarize_sleep <- function(data) {
    # YOUR CODE HERE
}


In [ ]:
group_means <- summarize_sleep(sleep_df)
group_means

In [ ]:
. = ottr::check("tests/q1_1.R")

The boxplots show the score distributions. The interaction plot connects sample means for each caffeine-cutoff group.


In [ ]:
# Grouped boxplot
p1 <- ggplot(sleep_df, aes(x = blue_light, y = sleep_score, fill = caffeine_cutoff)) +
        geom_boxplot() +
        labs(title = "Sleep Quality by Intervention Group", x = "Blue Light Filter",
            y = "Sleep Score", fill = "Caffeine Cutoff") + theme_bw() + theme(legend.position = "bottom")

# Interaction line plot of group means
p2 <- ggplot(group_means, aes(x = blue_light, y = mean_sleep_score,  group = caffeine_cutoff, color = caffeine_cutoff)) +
        geom_point(size = 3) + geom_line(linewidth = 1) +
        labs(title = "Interaction Plot: Mean Sleep Score", x = "Blue Light Filter",
            y = "Mean Sleep Score", color = "Caffeine Cutoff") + theme_bw() + theme(legend.position = "bottom")

print(p1)
print(p2)

**Question 1.2.** Let $\Delta_\mathrm{caffeine}$ be the population mean-score difference from adding a blue light filter when the caffeine cutoff is used, and let $\Delta_\mathrm{no\ caffeine}$ be the corresponding difference without the cutoff.

What constraint does no population interaction place on their difference, $\delta_\mathrm{cf}=\Delta_\mathrm{caffeine}-\Delta_\mathrm{no\ caffeine}$?

1. The blue-filter effect is zero in both caffeine groups.
2. The caffeine-cutoff effect is zero in both blue-filter groups.
3. The blue-filter effect is the same in both caffeine groups, so $\delta_\mathrm{cf}=0$.
4. The sum of the separate intervention effects is zero.

Set `no_interaction_delta` to the number of the correct choice.


_Points:_ 0.25

In [ ]:
no_interaction_delta <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_2.R")

**Question 1.3.** What does the nonparallel pattern in the sample interaction plot describe?

1. The caffeine groups differ by the same mean-score amount at both blue-filter settings.
2. The blue-filter difference averaged over the two caffeine groups is zero.
3. The within-group score spread changes with the caffeine cutoff.
4. The mean blue-filter difference changes with the caffeine-cutoff setting.

Set `non_parallel_interp` to one number.


_Points:_ 0.25

In [ ]:
non_parallel_interp <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_3.R")

Use the interaction model
$$\widehat{y}=\widehat{\beta}_0+\widehat{\beta}_1x_1+\widehat{\beta}_2x_2+\widehat{\beta}_3x_1x_2,\tag{1}$$
where $x_1$ indicates blue-filter use and $x_2$ indicates caffeine-cutoff use, with 0 for `"no"` and 1 for `"yes"`.


**Question 1.4.** Write `fit_sleep_model(data)` to fit the full interaction model to the supplied data, using `"no"` as the reference level for both factors. Return a list with `model`, the fitted `lm` object, and `coeffs`, its named coefficient vector.

The input has the same columns as `sleep_df` and includes all four treatment combinations. The fitted model must support `predict()` on new data with these column names. The supplied cell stores the results in `sleep_model` and `sleep_coeffs`.


_Points:_ 1

In [ ]:
fit_sleep_model <- function(data) {
    # YOUR CODE HERE
}


In [ ]:
sleep_fit <- fit_sleep_model(sleep_df)
sleep_model <- sleep_fit$model
sleep_coeffs <- sleep_fit$coeffs
summary(sleep_model)

In [ ]:
. = ottr::check("tests/q1_4.R")

**Question 1.5.** What does $\widehat{\beta}_1$, labeled `blue_lightyes` in the default output, estimate?

1. The blue-filter mean difference when no caffeine cutoff is used.
2. The blue-filter mean difference when the caffeine cutoff is used.
3. The blue-filter mean difference averaged over both caffeine-cutoff settings.
4. The caffeine-cutoff mean difference when no blue filter is used.

Set `beta1_interpretation` to one number.


_Points:_ 0.5

In [ ]:
beta1_interpretation <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_5.R")

**Question 1.6.** What does the interaction coefficient $\widehat{\beta}_3$ estimate?

1. The blue-filter mean difference when the caffeine cutoff is used.
2. The mean-score difference between using both interventions and using neither.
3. The blue-filter mean difference averaged over both caffeine-cutoff settings.
4. The blue-filter mean difference with the cutoff minus that difference without the cutoff.

Set `beta3_interpretation` to one number.


_Points:_ 0.75

In [ ]:
beta3_interpretation <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_6.R")

**Question 1.7.** What does the intercept $\widehat{\beta}_0$ estimate?

1. The mean score across all four treatment groups.
2. The mean score for participants using neither intervention.
3. The mean score for participants using both interventions.
4. The mean score for participants using exactly one intervention.

Set `beta0_interpretation` to one number.


_Points:_ 0.25

In [ ]:
beta0_interpretation <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_7.R")

**Question 1.8.** Write `filter_effect(model, cutoff)` to return the fitted mean-score difference from blue-filter use (`"yes"` minus `"no"`) at the specified caffeine-cutoff setting. `cutoff` is `"no"` or `"yes"`; return one number.

Handle other fitted models and changes to either factor's reference level. The supplied cell uses the caffeine cutoff and stores the result in `bl_effect_with_cc`.


_Points:_ 1

In [ ]:
filter_effect <- function(model, cutoff) {
    # YOUR CODE HERE
}


In [ ]:
bl_effect_with_cc <- filter_effect(sleep_model, "yes")
bl_effect_with_cc

In [ ]:
. = ottr::check("tests/q1_8.R")

For the original coding, the fitted blue-filter difference with the caffeine cutoff can be written as $\widehat{\beta}_1+\widehat{\beta}_3$.


**Question 1.9.** In the unweighted full interaction model, what is the sum of the residuals $e_i=y_i-\widehat y_i$ **within each treatment group**, apart from numerical rounding?

1. It can differ from zero by group, even though the overall residual sum is zero.
2. It equals the difference between that group's sample mean and the overall mean.
3. It is zero, because each fitted group mean matches its sample mean.
4. It is nonzero in groups where the interaction changes the fitted mean.

Set `residual_sum_interp` to one number.


_Points:_ 0.5

In [ ]:
residual_sum_interp <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_9.R")

## Part 2: Conditional Effects and Joint Tests

Use the sleep model to distinguish three questions: whether any treatment combination matters, whether the blue filter contributes at either caffeine setting, and whether its effect differs between settings. Then compare how the individual and joint tests behave across repeated samples.


In [ ]:
summary(sleep_model)


**Question 2.1.** Which null hypotheses describe the overall F-test in `summary(sleep_model)`? Set `overall_ftest` to the numbers of all correct choices.

1. $H_0:\beta_0=\beta_1=\beta_2=\beta_3=0$.
2. $H_0:\beta_1=\beta_2=\beta_3=0$.
3. $H_0:\mu_{00}=\mu_{01}=\mu_{10}=\mu_{11}$, where the subscripts give the two treatment settings.
4. $H_0:\beta_3=0$.
5. $H_0:\beta_1=\beta_3=0$.


_Points:_ 0.5

In [ ]:
overall_ftest <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_1.R")

**Question 2.2.** A researcher wants to test whether the blue filter has zero mean effect at **both** caffeine-cutoff settings. Which null hypothesis answers that question? Set `filter_contribution_null` to one number.

1. $H_0:\beta_1=0$.
2. $H_0:\beta_3=0$.
3. $H_0:\beta_1=\beta_3=0$.
4. $H_0:\beta_1+\beta_3=0$.


_Points:_ 0.5

In [ ]:
filter_contribution_null <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_2.R")

**Question 2.3.** Write `sleep_partial_test(data, drop)` to compare the full model `sleep_score ~ blue_light * caffeine_cutoff` with the appropriate reduced model.

`drop` is `"blue_light"`, `"caffeine_cutoff"`, or `"interaction"`. For a predictor, remove every term involving it. For `"interaction"`, remove only the interaction term.

Return a list with `full` and `reduced` fitted models, `df_num` (the number of coefficients removed), and the partial F-test `p_value`. Use the supplied data frame, with factor levels `"no"` and `"yes"`. The next cell applies your function to all three comparisons.


_Points:_ 2

In [ ]:
sleep_partial_test <- function(data, drop) {
    # YOUR CODE HERE
}


In [ ]:
blue_test <- sleep_partial_test(sleep_df, drop="blue_light")
caffeine_test <- sleep_partial_test(sleep_df, drop="caffeine_cutoff")
interaction_test <- sleep_partial_test(sleep_df, drop="interaction")
pval_bl_joint <- blue_test$p_value
pval_cc_joint <- caffeine_test$p_value
pval_no_interaction <- interaction_test$p_value
data.frame(comparison=c("Blue filter and its interaction", "Caffeine cutoff and its interaction", "Interaction only"),
    df_num=c(blue_test$df_num, caffeine_test$df_num, interaction_test$df_num),
    p_value=c(pval_bl_joint, pval_cc_joint, pval_no_interaction))


In [ ]:
. = ottr::check("tests/q2_3.R")

**Question 2.4.** The partial F-test that removes the interaction and the individual test for its coefficient address the same null hypothesis. Which one? Set `ftest_vs_ttest` to one number.

1. $H_0:\beta_1=0$: no blue-filter effect when the caffeine cutoff is absent.
2. $H_0:\beta_2=0$: no caffeine-cutoff effect when the blue filter is absent.
3. $H_0:\beta_1=\beta_2=\beta_3=0$: equal means in all four treatment groups.
4. $H_0:\beta_3=0$: the blue-filter effect is the same at both caffeine-cutoff settings.


_Points:_ 0.5

In [ ]:
ftest_vs_ttest <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_4.R")

### Two populations, two questions

Now use numeric 0/1 predictors and Normal errors. In Scenario A, the blue-filter effect is 2 at both caffeine settings. In Scenario B, it is zero without the cutoff and 3 with it. Both scenarios use a caffeine main coefficient of 2 and error SD of 2.

An individual coefficient test remains valid for its own null under the model conditions. It may answer a narrower question than the joint test.


In [ ]:
set.seed(152)
n_s <- 50
x1 <- rep(c(0, 1), each = 2 * n_s)
x2 <- rep(rep(c(0, 1), each = n_s), 2)
scen_A <- data.frame(x1 = x1, x2 = x2,
    y = 3 + 2 * x1 + 2 * x2 + rnorm(4 * n_s, sd = 2))
scen_B <- data.frame(x1 = x1, x2 = x2,
    y = 3 + 2 * x2 + 3 * x1 * x2 + rnorm(4 * n_s, sd = 2))
print(aggregate(y ~ x1 + x2, data = scen_A, FUN = mean))
print(aggregate(y ~ x1 + x2, data = scen_B, FUN = mean))


**Question 2.5.** Write `fit_factorial(data)` to fit `y ~ x1 * x2` for numeric 0/1 predictors. Return a list with `model`, the fitted `lm` object, and `p_value`, the two-sided p-value for `x1` in that model. The fitted model should support `predict()` on new data containing the same columns.

The supplied calls compare the two scenarios.


_Points:_ 0.75

In [ ]:
fit_factorial <- function(data) {
    # YOUR CODE HERE
}


In [ ]:
scenario_fit_A <- fit_factorial(scen_A)
scenario_fit_B <- fit_factorial(scen_B)
lm_scen_A <- scenario_fit_A$model
lm_scen_B <- scenario_fit_B$model
pval_x1_A <- scenario_fit_A$p_value
pval_x1_B <- scenario_fit_B$p_value
data.frame(Scenario=c("A", "B"),
    baseline_x1_pvalue=c(pval_x1_A, pval_x1_B),
    interaction_pvalue=c(summary(lm_scen_A)$coefficients["x1:x2", "Pr(>|t|)"],
        summary(lm_scen_B)$coefficients["x1:x2", "Pr(>|t|)"]))


In [ ]:
. = ottr::check("tests/q2_5.R")

**Question 2.6.** Which individual p-value directly tests whether the blue-filter effect differs between caffeine-cutoff settings? Set `relevant_pvals_B` to one number.

1. The p-value for `x1:x2`.
2. The p-value for `x2`.
3. The p-value for the intercept.
4. The p-value for `x1`.


_Points:_ 0.5

In [ ]:
relevant_pvals_B <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_6.R")

### Rejection rates over repeated samples

Use a balanced design with `n_per_group` observations at each 0/1 predictor combination:
$$y=3+\beta_1x_1+\beta_2x_2+\beta_3x_1x_2+\varepsilon,
\qquad \varepsilon\sim N(0,\sigma^2),$$
with independent errors. Compare the individual null $\beta_1=0$ with the joint null $\beta_1=\beta_3=0$. A rejection rate estimates Type I error when its own null is true and power when it is false.


**Question 2.7.** Write `sim_test_rates(n_per_group, beta1, beta2, beta3, sigma = 2.5, n_sim = 500, alpha = 0.05)` for this model.

Estimate the rejection rates of the two tests using fresh simulated data in each replicate. Use the full interaction model and the same dataset for both tests, with all supplied parameter values.

Return a named numeric vector of rejection **fractions**, with entries `ttest_rej` and `ftest_rej`.


_Points:_ 3

In [ ]:
sim_test_rates <- function(n_per_group, beta1, beta2, beta3, sigma = 2.5, n_sim = 500, alpha = 0.05) {
    # YOUR CODE HERE
}


In [ ]:
set.seed(123)
sanity_check_ftest <- sim_test_rates(n_per_group=25, beta1=0, beta2=1, beta3=0,
    sigma=2.5, n_sim=500, alpha=0.05)
sanity_check_ftest


In [ ]:
. = ottr::check("tests/q2_7.R")

Run the supplied comparisons. The settings change the main effect and interaction; the simulation results are estimates and will vary across runs.


In [ ]:
set.seed(123)
result_scen1 <- sim_test_rates(n_per_group=50, beta1=2, beta2=1, beta3=0, sigma=2.5, n_sim=1000, alpha=0.05)
result_scen2 <- sim_test_rates(n_per_group=50, beta1=1.2, beta2=1, beta3=1.5, sigma=2.5, n_sim=1000, alpha=0.05)
result_scen3 <- sim_test_rates(n_per_group=50, beta1=0, beta2=1, beta3=3, sigma=2.5, n_sim=1000, alpha=0.05)
sim_results_combined <- data.frame(
    Scenario = rep(c("1: beta1=2, beta3=0", "2: beta1=1.2, beta3=1.5", "3: beta1=0, beta3=3"), each=2),
    Test = rep(c("Individual t-test", "Joint F-test"), 3),
    Rejection_rate = as.numeric(c(result_scen1[c("ttest_rej", "ftest_rej")],
        result_scen2[c("ttest_rej", "ftest_rej")], result_scen3[c("ttest_rej", "ftest_rej")])))
print(sim_results_combined, row.names = FALSE)
local({
    old_par <- par(mar = c(5, 4.5, 3, 1))
    on.exit(par(old_par))
    bars <- barplot(matrix(sim_results_combined$Rejection_rate, nrow=2), beside=TRUE,
        names.arg=c("Scenario 1", "Scenario 2", "Scenario 3"),
        col=c("steelblue", "coral"), ylim=c(0, 1.15),
        ylab="Estimated rejection rate", main="Two tests, different null hypotheses")
    abline(h=0.05, lty=2, col="gray40")
    legend("top", legend=c("Individual x1 t-test", "Joint x1 + interaction F-test"),
        fill=c("steelblue", "coral"), bty="n", horiz=TRUE, cex=0.8)
})


**Question 2.8.** Consider a new population with $\beta_1=1$ and $\beta_3=-1$; keep the other model settings unchanged.

* Set `new_filter_effects` to a numeric vector giving the blue-filter mean effect when the caffeine cutoff is absent, then when it is present.
* Set `new_test_types` to two codes: first for the individual test's rejection rate, then for the joint test's rate. Use **1** for Type I error and **2** for power.

Classify each rate using that test's own null hypothesis.


_Points:_ 1.25

In [ ]:
new_filter_effects <- NULL # YOUR CODE HERE
new_test_types <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_8.R")

The two rejection rates belong to different hypotheses. In the displayed Scenario 3, the individual null is true while the joint null is false. Comparing their bar heights does not establish that one test is generally better.


## Part 3: Comparing Slopes Between Groups

In this simulated study, 200 adults have a recorded daily step count, diet group, and resting heart rate. Use the continuous-by-binary interaction from Lecture 12 to compare the step slopes of the two diet groups.

Treat the fitted differences as associations, rather than causal diet effects.


In [ ]:
set.seed(123)
n_heart <- 200
daily_steps <- round(runif(n_heart, min = 2000, max = 15000))
on_diet <- sample(c("no", "yes"), n_heart, replace = TRUE)
heart_rate <- ifelse(on_diet == "no", 85 - 0.0012 * daily_steps,
                     88 - 0.0018 * daily_steps) + rnorm(n_heart, sd = 3)
heart_df <- data.frame(heart_rate = round(heart_rate, 1), daily_steps = daily_steps,
                       on_diet = factor(on_diet, levels = c("no", "yes")))
head(heart_df)


In [ ]:
ggplot(heart_df, aes(daily_steps, heart_rate, color = on_diet, shape = on_diet)) +
    geom_point(alpha = 0.65) + geom_smooth(method = "lm", se = FALSE, linewidth = 1) +
    labs(title = "Heart rate and daily steps", x = "Average daily steps",
         y = "Resting heart rate (bpm)", color = "On diet", shape = "On diet") + theme_bw()


**Question 3.1.** Write `fit_heart_model(data, interaction = FALSE)` to fit heart rate using daily steps and diet group. Return an additive `lm` model when `interaction = FALSE` and an interaction model when `interaction = TRUE`.

Use the columns `heart_rate`, `daily_steps`, and `on_diet`, with `no` as the reference group even if the input factor levels arrive in the other order. The supplied cell fits the additive model and displays its diet coefficient.


_Points:_ 0.5

In [ ]:
fit_heart_model <- function(data, interaction = FALSE) {
    # YOUR CODE HERE
}


In [ ]:
hr_model_no_interaction <- fit_heart_model(heart_df)
diet_coef_no_interaction <- coef(hr_model_no_interaction)["on_dietyes"]
summary(hr_model_no_interaction)


In [ ]:
. = ottr::check("tests/q3_1.R")

**Question 3.2.** What does the additive model assume about steps and heart rate? Set `no_interaction_assumption` to one number.

1. The step slope is zero in the diet group.
2. Both groups share a step slope, so their fitted lines are parallel.
3. The diet group has a higher intercept.
4. Each group has its own step slope.


_Points:_ 0.25

In [ ]:
no_interaction_assumption <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q3_2.R")

In [ ]:
hr_model_with_interaction <- fit_heart_model(heart_df, interaction = TRUE)
summary(hr_model_with_interaction)


**Question 3.3.** Write `step_effects(model, step_change = 1000)` to return the fitted mean heart-rate change for an increase of `step_change` steps in each group. Return a named numeric vector with `no_diet` and `on_diet`, in that order.

Handle different step increments and fitted models, including either diet reference group and either order of the predictor terms. The cell below evaluates a 1,000-step increase.


_Points:_ 0.75

In [ ]:
step_effects <- function(model, step_change = 1000) {
    # YOUR CODE HERE
}


In [ ]:
thousand_step_effects <- step_effects(hr_model_with_interaction)
thousand_step_effects


In [ ]:
. = ottr::check("tests/q3_3.R")

**Question 3.4.** Write `joint_diet_test(data)` to test the diet indicator and its interaction with steps together. Return a list with the full interaction model as `full`, the steps-only model as `null`, and the partial F-test p-value as `p_value`.

In $E(y\mid x,d)=\beta_0+\beta_1x+\beta_2d+\beta_3xd$, where $d=1$ denotes the diet group, this tests $H_0:\beta_2=\beta_3=0$.


_Points:_ 1

In [ ]:
joint_diet_test <- function(data) {
    # YOUR CODE HERE
}


In [ ]:
diet_joint_result <- joint_diet_test(heart_df)
hr_null_model <- diet_joint_result$null
pval_diet_joint <- diet_joint_result$p_value
pval_diet_joint


In [ ]:
. = ottr::check("tests/q3_4.R")

**Question 3.5.** When $\widehat\beta_3\ne0$, at what step count do the two fitted mean lines intersect? Set `steps_equivalence_exp` to one number. An intersection outside the observed step range would require extrapolation.

1. $-\widehat\beta_2/\widehat\beta_3$
2. $\widehat\beta_1/\widehat\beta_3$
3. $-(\widehat\beta_0+\widehat\beta_2)/(\widehat\beta_1+\widehat\beta_3)$
4. $\widehat\beta_2/\widehat\beta_3$


_Points:_ 0.25

In [ ]:
steps_equivalence_exp <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q3_5.R")

**Question 3.6.** Write `diet_difference(model, steps)` to return the fitted mean heart rate for the diet group minus that for the no-diet group at the supplied step count. Return one number.

Your function should work at different step counts and with different fitted interaction models, including either diet reference group. The supplied cell compares the groups at 10,000 steps.


_Points:_ 0.75

In [ ]:
diet_difference <- function(model, steps) {
    # YOUR CODE HERE
}


In [ ]:
diff_10k <- diet_difference(hr_model_with_interaction, 10000)
diet_intercept_difference <- unname(coef(hr_model_with_interaction)["on_dietyes"])
data.frame(daily_steps = c(0, 10000),
           diet_minus_no_diet_bpm = c(diet_intercept_difference, diff_10k))


In [ ]:
. = ottr::check("tests/q3_6.R")

Curvature or changing spread in the residual plot below would suggest a problem with the fitted model.


In [ ]:
res_df <- data.frame(fitted = fitted(hr_model_with_interaction),
                     residual = residuals(hr_model_with_interaction))
ggplot(res_df, aes(fitted, residual)) + geom_point(alpha = 0.6) +
    geom_hline(yintercept = 0, linetype = 2) +
    labs(title = "Interaction model residuals", x = "Fitted heart rate (bpm)", y = "Residual (bpm)") +
    theme_minimal()


**Question 3.7.** Adapt Lecture 3's percentile-bootstrap procedure to estimate uncertainty in the interaction coefficient. Write `bootstrap_interaction(data, reps = 1000, level = 0.95)` to resample whole rows with replacement, fit the interaction model in each replicate, and return:

* `estimates`: the numeric vector of interaction estimates, one per replicate;
* `ci`: the percentile interval at the supplied confidence level, lower endpoint first.

Use `no` as the reference group, regardless of the input factor order. Keep each row's steps, diet, and heart-rate measurements together. This procedure assumes independent sampled observations; it does not repair a misspecified mean model or dependent data.


_Points:_ 2

In [ ]:
bootstrap_interaction <- function(data, reps = 1000, level = 0.95) {
    # YOUR CODE HERE
}


In [ ]:
set.seed(123)
interaction_bootstrap <- bootstrap_interaction(heart_df)
boot_dist <- interaction_bootstrap$estimates
interaction_boot_ci <- interaction_bootstrap$ci
data.frame(unit = c("bpm per step", "bpm per 1,000 steps"),
           lower = unname(interaction_boot_ci[1]) * c(1, 1000),
           upper = unname(interaction_boot_ci[2]) * c(1, 1000))


In [ ]:
. = ottr::check("tests/q3_7.R")

**Question 3.8.** Use the bootstrap interval and the fitted comparisons above. Set `bootstrap_interpretation` to the numbers of all correct statements.

1. The interval estimates uncertainty in the diet-group slope minus the no-diet slope, measured in bpm per additional daily step.
2. Multiplying its endpoints by 1,000 gives an interval for the diet group's own mean heart-rate change over a 1,000-step increase.
3. The 95% interval excludes zero, providing evidence that the two groups have different step slopes.
4. The positive diet coefficient in the interaction model contradicts `diff_10k < 0`, since both compare dieting with non-dieting adults.
5. The negative interaction means the fitted diet-minus-no-diet gap decreases as steps increase; its value at a particular step count also depends on the diet coefficient.


_Points:_ 0.5

In [ ]:
bootstrap_interpretation <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q3_8.R")

The diet coefficient in the interaction model compares the groups at zero steps, outside the observed range. `diff_10k` compares them at 10,000 steps. With different slopes, the two comparisons need not have the same sign.


## Congratulations, you are finished!!

To submit your assignment:

1. Restart the kernel and run all cells from top to bottom. In Jupyter, use `Kernel -> Restart Kernel and Run All Cells...` or the equivalent command in your interface.
2. Check the full notebook for errors and confirm that every grader check passes. If a check fails, fix the earliest error first, then rerun from the beginning.
3. Save the notebook, download the `.ipynb` file using `File -> Download` (or the equivalent option), and upload it to Gradescope.
4. Wait for Gradescope to finish grading. Confirm that all tests pass there as well; passing the notebook checks alone does not confirm a successful submission.
5. Verify that Gradescope shows your submission and keep the confirmation email, if provided, as proof of submission.